# PoWR + ATLAS/Phoenix fitting to Gaia BP/RP spectra
# v4 - Polynomial Spectral Model (PSM) Refinement

**Algorithm (based on Rix et al. 2016, ApJL 826, L25):**

1. **Full brute-force search**: Evaluate all ~660 models to find best discrete (Teff*, logg*, A_V*, R_V*)
2. **Build local PSM**: Fit 2D quadratic to intrinsic spectra AND auxiliary quantities in 3×3 neighborhood
3. **Continuous optimization**: Minimize χ² over (Teff, logg, A_V, R_V) using PSM + analytic extinction
4. **Interpolate auxiliary quantities**: Compute logL, Mass via PSM; derive R_Rsun via Stefan-Boltzmann

**Key equations (per wavelength / quantity):**
```
log(f_intrinsic(λ; T, g)) ≈ a₀ + a₁T̃ + a₂g̃ + a₃T̃² + a₄g̃² + a₅T̃g̃
logL(T, g) ≈ b₀ + b₁T̃ + b₂g̃ + b₃T̃² + b₄g̃² + b₅T̃g̃
Mass(T, g) ≈ c₀ + c₁T̃ + c₂g̃ + c₃T̃² + c₄g̃² + c₅T̃g̃
R_Rsun = sqrt(L / (4πσT⁴)) / R_sun  [Stefan-Boltzmann]
```
where T̃, g̃ are normalized coordinates within the local patch.

**Extinction applied analytically:**
```
f_obs = scale × f_intrinsic × 10^(-0.4 × A_V × k(λ, R_V))
```

**Changes from v2:**
- Adds PSM refinement step for continuous parameter estimation
- **NEW**: Interpolates logL, Mass; derives R_Rsun from Stefan-Boltzmann (no more discretization)
- Returns both discrete best-fit and continuous refined parameters
- Deduplicates neighbors to handle manifest duplicate bug
- Same model filtering (Teff > 40kK & logg ≥ 4.0)
- Same input column preservation

**Output:**
- CSV with all input catalog columns plus fit parameters
- `*_fit` columns: PSM-interpolated continuous values
- `*_grid` columns: Best discrete grid values (for comparison)

In [3]:
!pip install dust_extinction

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from pathlib import Path
import pandas as pd
import glob
import re
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import minimize
import time

# Check for dust_extinction package
try:
    from dust_extinction.parameter_averages import G23
    try:
        from dust_extinction.parameter_averages import G24
    except ImportError:
        G24 = None
    print("dust_extinction package loaded successfully")
except ImportError:
    raise ImportError(
        "Please install dust_extinction >= 1.2:\n"
        "    pip install dust_extinction\n"
    )

dust_extinction package loaded successfully


In [5]:
# ========================== CONFIGURATION ==========================
POWR_MODEL_DIR   = Path('./griddl-gal-ob-vd3-line_calib')
ATLAS_MODEL_DIR  = Path('./stellar_models')
MODEL_MANIFEST   = Path('./model_manifest.csv')
CATALOG_FITS     = 'IACOB_all_enriched.fits'  # Use enriched catalog
SPECTRA_DIR      = Path('./BPRP_spectra')
PLOT_DIR         = Path('./fit_plots_v4')
PLOT_DIR.mkdir(exist_ok=True)

# Auto-generate output CSV name from input catalog
OUTPUT_CSV       = Path(CATALOG_FITS).stem + '_fits_v4.csv'
print(f"Output will be: {OUTPUT_CSV}")

# ============== TEST MODE ==============
N_MAX_FIT = None  # Set to None to fit all stars, or a number for testing

# ============== PRIOR CONFIGURATION ==============
USE_AV_PRIOR = True
AV_PRIOR_WEIGHT = 10.0
USE_TEFF_PRIOR = False
TEFF_PRIOR_WEIGHT = 1.0
TEFF_PRIOR_TEFF_MAX = 7500
TEFF_PRIOR_SIGMA = 500

# Fitting grid - R_V must be in [2.3, 5.6] for G23
A_V_GRID  = np.arange(0.0, 7.1, 0.1)
R_V_GRID  = np.array([2.3, 2.5, 2.8, 3.1, 3.4, 3.7, 4.0, 4.5, 5.0, 5.5])
WAVELENGTH_FIT_MIN = 340.0
WAVELENGTH_FIT_MAX = 900.0
SYSTEMATIC_FLOOR   = 0.03
BLUE_WEIGHT_REGION = (340.0, 480.0)
BLUE_WEIGHT_FACTOR = 2.0

# PSM configuration
PSM_NEIGHBOR_GRID = 3  # 3x3 grid around best model

W_M2_NM_TO_ERG_S_CM2_A = 1e2

# Check manifest
USE_MANIFEST = MODEL_MANIFEST.exists()
if USE_MANIFEST:
    print(f"Model manifest found: {MODEL_MANIFEST}")
else:
    print(f"No manifest at {MODEL_MANIFEST} - will use glob-based loading")

print(f"\nPrior settings:")
print(f"  A_V prior: {'ON' if USE_AV_PRIOR else 'OFF'} (weight={AV_PRIOR_WEIGHT})")
print(f"  Teff prior: {'ON' if USE_TEFF_PRIOR else 'OFF'}")
print(f"\nR_V grid: {list(R_V_GRID)}")
print(f"\nPSM: Will use {PSM_NEIGHBOR_GRID}x{PSM_NEIGHBOR_GRID} local grid for refinement")

if N_MAX_FIT is not None:
    print(f"\nTEST MODE: Will fit only first {N_MAX_FIT} stars")

Output will be: IACOB_all_enriched_fits_v4.csv
Model manifest found: model_manifest.csv

Prior settings:
  A_V prior: ON (weight=10.0)
  Teff prior: OFF

R_V grid: [np.float64(2.3), np.float64(2.5), np.float64(2.8), np.float64(3.1), np.float64(3.4), np.float64(3.7), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.5)]

PSM: Will use 3x3 local grid for refinement


In [6]:
# ========================== LOAD CATALOG ==========================
# Load using astropy Table to preserve all columns
print("Loading catalog...")
catalog_table = Table.read(CATALOG_FITS)
catalog_df = catalog_table.to_pandas()

# Convert source_id to int64 if needed
if catalog_df['source_id'].dtype == object:
    catalog_df['source_id'] = catalog_df['source_id'].astype(np.int64)

# Extract arrays for fitting loop
source_ids = catalog_df['source_id'].values
gmag = catalog_df['Gmag'].values
parallax = catalog_df['parallax'].values if 'parallax' in catalog_df.columns else catalog_df['Plx'].values
parallax_error = catalog_df['parallax_error'].values if 'parallax_error' in catalog_df.columns else np.full(len(source_ids), 0.1)

# Load enrichment columns
Teff_A23_catalog = catalog_df['Teff_A23'].values if 'Teff_A23' in catalog_df.columns else np.full(len(source_ids), np.nan)
A_V_W25_catalog = catalog_df['A_V_W25'].values if 'A_V_W25' in catalog_df.columns else np.full(len(source_ids), np.nan)
A_V_W25_err_catalog = catalog_df['A_V_W25_err'].values if 'A_V_W25_err' in catalog_df.columns else np.full(len(source_ids), np.nan)

print(f"Loaded {len(source_ids)} sources")
print(f"Catalog columns: {list(catalog_df.columns)}")

Loading catalog...
Loaded 280 sources
Catalog columns: ['source_id', 'parallax', 'parallax_error', 'Gmag', 'ra', 'dec', 'Teff_A23', 'logg_A23', 'MH_A23', 'EBV_W25', 'EBV_W25_err', 'A_V_W25', 'A_V_W25_err', 'dist_max_W25']


In [7]:
# ============================================================================
# LOAD ALL MODELS (from manifest)
# ============================================================================

all_models = []

if USE_MANIFEST:
    print(f"\nLoading models from manifest: {MODEL_MANIFEST}")
    
    manifest_df = pd.read_csv(MODEL_MANIFEST)
    print(f"  Manifest contains {len(manifest_df)} models")
    
    for src in manifest_df['source'].unique():
        subset = manifest_df[manifest_df['source'] == src]
        print(f"  {src}: {len(subset)} models, Teff={subset['Teff'].min()}-{subset['Teff'].max()}K")
    
    for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Loading models"):
        source = row['source']
        filename = row['filename']
        
        if source == 'PoWR':
            filepath = POWR_MODEL_DIR / filename
        else:
            filepath = ATLAS_MODEL_DIR / filename
        
        if not filepath.exists():
            continue
        
        try:
            data = np.loadtxt(filepath)
            wave_model = data[:, 0]
            log_flux_model = data[:, 1]
            flux_model = 10**log_flux_model
            
            all_models.append({
                'Teff': row['Teff'],
                'logg': row['logg'],
                'source': source,
                'logL': row['logL'],
                'Mass': row['Mass'],
                'Mass_std': row.get('Mass_std', np.nan),
                'R_Rsun': row['R_Rsun'],
                'wavelength': wave_model,
                'flux': flux_model,
                'filename': filename
            })
        except Exception as e:
            print(f"  Error loading {filepath}: {e}")
    
    print(f"\nSuccessfully loaded {len(all_models)} models from manifest")

else:
    raise NotImplementedError("Please run download notebook to generate model_manifest.csv")

powr_models = all_models
print(f"\nModel library ready: {len(powr_models)} models")


Loading models from manifest: model_manifest.csv
  Manifest contains 471 models
  ATLAS: 39 models, Teff=10000-15000K
  PHOENIX: 192 models, Teff=3800-10000K
  PoWR: 240 models, Teff=15000-56000K


Loading models: 100%|█████████████████████████████████████████████| 471/471 [00:05<00:00, 82.53it/s]


Successfully loaded 471 models from manifest

Model library ready: 471 models


In [8]:
# ========================== GAIA GRID & RESAMPLE ==========================
ref_file = next(SPECTRA_DIR.glob('*.fits'))
with fits.open(ref_file) as hdul:
    gaia_wavelength_nm = np.asarray(hdul[1].data['wavelength']).flatten()
gaia_wavelength_A = gaia_wavelength_nm * 10

diff = np.diff(gaia_wavelength_A)
bins = np.concatenate([[gaia_wavelength_A[0]-diff[0]/2],
                       gaia_wavelength_A[:-1] + diff/2,
                       [gaia_wavelength_A[-1]+diff[-1]/2]])

print("Resampling models to Gaia grid...")
for m in tqdm(powr_models):
    flux = np.zeros(len(gaia_wavelength_A))
    for i in range(len(gaia_wavelength_A)):
        mask = (m['wavelength'] >= bins[i]) & (m['wavelength'] < bins[i+1])
        if mask.sum()>0:
            flux[i] = np.mean(m['flux'][mask])
        else:
            flux[i] = np.interp(gaia_wavelength_A[i], m['wavelength'], m['flux'])
    flux = gaussian_filter1d(flux, sigma=2.0)
    m['flux_gaia'] = flux
    m['wave_gaia'] = gaia_wavelength_A

Resampling models to Gaia grid...


100%|█████████████████████████████████████████████████████████████| 471/471 [00:22<00:00, 20.52it/s]


In [9]:
# ============================================================================
# VECTORIZATION + MODEL FILTERING
# ============================================================================

print("Stacking models for vectorized computation...")

# Model filtering: Remove unphysical PoWR models (Teff > 40kK AND logg >= 4.0)
TEFF_LOGG_CUT_TEFF = 40000  # K
TEFF_LOGG_CUT_LOGG = 4.0    # dex

MODEL_TEFF_RAW = np.array([m['Teff'] for m in powr_models])
MODEL_LOGG_RAW = np.array([m['logg'] for m in powr_models])
MODEL_SOURCE_RAW = np.array([m['source'] for m in powr_models])

unphysical_mask = (MODEL_SOURCE_RAW == 'PoWR') & (MODEL_TEFF_RAW > TEFF_LOGG_CUT_TEFF) & (MODEL_LOGG_RAW >= TEFF_LOGG_CUT_LOGG)
valid_mask = ~unphysical_mask

n_excluded = np.sum(unphysical_mask)
print(f"\nModel filtering: Excluding {n_excluded} unphysical PoWR models")
print(f"  (Teff > {TEFF_LOGG_CUT_TEFF}K AND logg >= {TEFF_LOGG_CUT_LOGG})")

powr_models_filtered = [m for i, m in enumerate(powr_models) if valid_mask[i]]
print(f"  Before: {len(powr_models)} models")
print(f"  After:  {len(powr_models_filtered)} models\n")

powr_models = powr_models_filtered

n_models = len(powr_models)
n_wavelengths = len(gaia_wavelength_A)

# Stack model fluxes: shape (n_models, n_wavelengths)
MODEL_FLUX_ALL = np.zeros((n_models, n_wavelengths))
for i, m in enumerate(powr_models):
    MODEL_FLUX_ALL[i, :] = m['flux_gaia']

# Stack model properties
MODEL_TEFF = np.array([m['Teff'] for m in powr_models])
MODEL_LOGG = np.array([m['logg'] for m in powr_models])
MODEL_LOGL = np.array([m['logL'] for m in powr_models])
MODEL_MASS = np.array([m['Mass'] for m in powr_models])
MODEL_MASS_STD = np.array([m.get('Mass_std', np.nan) for m in powr_models])
MODEL_R_RSUN = np.array([m['R_Rsun'] for m in powr_models])
MODEL_SOURCE = np.array([m['source'] for m in powr_models])
MODEL_FILENAME = np.array([m['filename'] for m in powr_models])

print(f"  Model flux array shape: {MODEL_FLUX_ALL.shape}")
print(f"  Memory usage: {MODEL_FLUX_ALL.nbytes / 1e6:.1f} MB")

# Build lookup for finding neighbors in (Teff, logg) space
# Create unique (Teff, logg) pairs and their indices
UNIQUE_TEFF = np.sort(np.unique(MODEL_TEFF))
UNIQUE_LOGG = np.sort(np.unique(MODEL_LOGG))
print(f"\nUnique Teff values: {len(UNIQUE_TEFF)} ({UNIQUE_TEFF.min():.0f} - {UNIQUE_TEFF.max():.0f} K)")
print(f"Unique logg values: {len(UNIQUE_LOGG)} ({UNIQUE_LOGG.min():.1f} - {UNIQUE_LOGG.max():.1f})")

# Report model breakdown by source
for src in ['PoWR', 'ATLAS', 'PHOENIX']:
    n_src = np.sum(MODEL_SOURCE == src)
    print(f"  {src}: {n_src} models")

Stacking models for vectorized computation...

Model filtering: Excluding 25 unphysical PoWR models
  (Teff > 40000K AND logg >= 4.0)
  Before: 471 models
  After:  446 models

  Model flux array shape: (446, 343)
  Memory usage: 1.2 MB

Unique Teff values: 72 (3800 - 49000 K)
Unique logg values: 15 (2.0 - 4.5)
  PoWR: 215 models
  ATLAS: 39 models
  PHOENIX: 192 models


In [10]:
# ============================================================================
# EXTINCTION FUNCTION - GORDON+ 2023
# ============================================================================

import astropy.units as u

if G24 is not None:
    ExtinctionModel = G24
    print("Using extinction model: G24 (Gordon+ 2024)")
else:
    ExtinctionModel = G23
    print("Using extinction model: G23 (Gordon+ 2023)")

def build_extinction_table(wavelength_aa, R_V_grid):
    """Pre-compute A(lambda)/A_V for all R_V values."""
    table = {}
    wave_with_units = wavelength_aa * u.AA
    for Rv in R_V_grid:
        ext = ExtinctionModel(Rv=Rv)
        table[Rv] = ext(wave_with_units)
    return table

# Build for discrete grid
print(f"Building {ExtinctionModel.__name__} extinction lookup table...")
EXT_TABLE = build_extinction_table(gaia_wavelength_A, R_V_GRID)
print(f"   Extinction table ready for R_V = {list(R_V_GRID)}")

# Build stacked array for vectorized operations
EXT_ARRAY = np.zeros((len(R_V_GRID), len(gaia_wavelength_A)))
for i, Rv in enumerate(R_V_GRID):
    EXT_ARRAY[i, :] = EXT_TABLE[Rv]
print(f"   Extinction array shape: {EXT_ARRAY.shape}")

def get_extinction_curve(R_V, wavelength_aa=gaia_wavelength_A):
    """Get A(lambda)/A_V for arbitrary R_V (for PSM optimization)."""
    ext = ExtinctionModel(Rv=R_V)
    return ext(wavelength_aa * u.AA)

def apply_reddening(wavelength_aa, flux, A_V, R_V=3.1):
    """Apply Gordon+ extinction to a spectrum."""
    if R_V in EXT_TABLE:
        A_lambda_over_Av = EXT_TABLE[R_V]
    else:
        A_lambda_over_Av = get_extinction_curve(R_V, wavelength_aa)
    A_lambda = A_lambda_over_Av * A_V
    return flux * 10**(-0.4 * A_lambda)

def calculate_A_G(A_V, R_V=3.1):
    """Compute A_G (Gaia G band) from A_V."""
    lambda_G_aa = 6420.0
    ext = ExtinctionModel(Rv=R_V)
    A_lambda_over_Av = ext(lambda_G_aa * u.AA)
    return float(A_lambda_over_Av) * A_V

def wavelength_weights(wavelength_nm, blue_region=BLUE_WEIGHT_REGION, blue_weight=BLUE_WEIGHT_FACTOR):
    """Return weights for chi-square."""
    weights = np.ones_like(wavelength_nm)
    blue_mask = (wavelength_nm >= blue_region[0]) & (wavelength_nm <= blue_region[1])
    weights[blue_mask] = blue_weight
    return weights

Using extinction model: G23 (Gordon+ 2023)
Building G23 extinction lookup table...
   Extinction table ready for R_V = [np.float64(2.3), np.float64(2.5), np.float64(2.8), np.float64(3.1), np.float64(3.4), np.float64(3.7), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.5)]
   Extinction array shape: (10, 343)


In [11]:
# ============================================================================
# POLYNOMIAL SPECTRAL MODEL (PSM) FUNCTIONS
# Based on Rix et al. 2016, ApJL 826, L25
# ============================================================================

# Physical constants for Stefan-Boltzmann
L_SUN = 3.828e33  # erg/s
R_SUN = 6.957e10  # cm
SIGMA_SB = 5.670374419e-5  # erg/cm^2/s/K^4

def find_neighbors_in_grid(best_idx, n_neighbors=1):
    """
    Find neighboring models in (Teff, logg) space around the best model.
    
    Parameters
    ----------
    best_idx : int
        Index of best model in MODEL_* arrays
    n_neighbors : int
        Number of neighbors in each direction (1 = 3x3 grid)
    
    Returns
    -------
    neighbor_indices : list of int
        Indices of neighboring models (including best model)
    teff_range : tuple
        (min, max) Teff of neighbor grid
    logg_range : tuple
        (min, max) logg of neighbor grid
    """
    best_teff = MODEL_TEFF[best_idx]
    best_logg = MODEL_LOGG[best_idx]
    best_source = MODEL_SOURCE[best_idx]
    
    # Find Teff and logg values for this source family only
    source_mask = MODEL_SOURCE == best_source
    source_teffs = np.sort(np.unique(MODEL_TEFF[source_mask]))
    source_loggs = np.sort(np.unique(MODEL_LOGG[source_mask]))
    
    # Find position of best model in the source grid
    teff_idx = np.searchsorted(source_teffs, best_teff)
    logg_idx = np.searchsorted(source_loggs, best_logg)
    
    # Handle edge cases - ensure we have n_neighbors on each side if possible
    teff_min_idx = max(0, teff_idx - n_neighbors)
    teff_max_idx = min(len(source_teffs), teff_idx + n_neighbors + 1)
    logg_min_idx = max(0, logg_idx - n_neighbors)
    logg_max_idx = min(len(source_loggs), logg_idx + n_neighbors + 1)
    
    # Get the Teff and logg values in the neighborhood
    neighbor_teffs = source_teffs[teff_min_idx:teff_max_idx]
    neighbor_loggs = source_loggs[logg_min_idx:logg_max_idx]
    
    # Find all model indices that fall within this neighborhood
    # Use a set to avoid duplicates from manifest bug
    neighbor_set = set()
    neighbor_indices = []
    for i in range(n_models):
        if MODEL_SOURCE[i] == best_source:
            if MODEL_TEFF[i] in neighbor_teffs and MODEL_LOGG[i] in neighbor_loggs:
                key = (MODEL_TEFF[i], MODEL_LOGG[i])
                if key not in neighbor_set:
                    neighbor_set.add(key)
                    neighbor_indices.append(i)
    
    teff_range = (neighbor_teffs.min(), neighbor_teffs.max())
    logg_range = (neighbor_loggs.min(), neighbor_loggs.max())
    
    return neighbor_indices, teff_range, logg_range


def build_psm_coefficients(neighbor_indices, teff_range, logg_range):
    """
    Build quadratic PSM coefficients for log(flux), logL, and Mass.
    
    Model: quantity = a0 + a1*T_norm + a2*g_norm + a3*T_norm^2 + a4*g_norm^2 + a5*T_norm*g_norm
    
    where T_norm, g_norm are normalized to [-1, 1] within the local patch.
    
    Parameters
    ----------
    neighbor_indices : list of int
        Indices of models to use for fitting
    teff_range : tuple
        (min, max) Teff for normalization
    logg_range : tuple
        (min, max) logg for normalization
    
    Returns
    -------
    psm_data : dict containing:
        'flux_coeffs': ndarray, shape (n_wavelengths, 6) - for log(flux)
        'logL_coeffs': ndarray, shape (6,) - for logL
        'mass_coeffs': ndarray, shape (6,) - for Mass
        'teff_norm': tuple (center, scale)
        'logg_norm': tuple (center, scale)
    """
    n_neighbors = len(neighbor_indices)
    
    # Normalization parameters
    teff_center = 0.5 * (teff_range[0] + teff_range[1])
    teff_scale = 0.5 * (teff_range[1] - teff_range[0]) if teff_range[1] > teff_range[0] else 1.0
    logg_center = 0.5 * (logg_range[0] + logg_range[1])
    logg_scale = 0.5 * (logg_range[1] - logg_range[0]) if logg_range[1] > logg_range[0] else 1.0
    
    # Build design matrix for quadratic fit
    # Columns: [1, T_norm, g_norm, T_norm^2, g_norm^2, T_norm*g_norm]
    X = np.zeros((n_neighbors, 6))
    for i, idx in enumerate(neighbor_indices):
        T_norm = (MODEL_TEFF[idx] - teff_center) / teff_scale
        g_norm = (MODEL_LOGG[idx] - logg_center) / logg_scale
        X[i, :] = [1, T_norm, g_norm, T_norm**2, g_norm**2, T_norm * g_norm]
    
    # ---- Fit log(flux) at each wavelength ----
    log_flux_neighbors = np.log10(MODEL_FLUX_ALL[neighbor_indices, :] + 1e-50)
    flux_coeffs, _, _, _ = np.linalg.lstsq(X, log_flux_neighbors, rcond=None)
    flux_coeffs = flux_coeffs.T  # shape (n_wavelengths, 6)
    
    # ---- Fit logL ----
    logL_neighbors = MODEL_LOGL[neighbor_indices]
    logL_coeffs, _, _, _ = np.linalg.lstsq(X, logL_neighbors, rcond=None)
    
    # ---- Fit Mass ----
    mass_neighbors = MODEL_MASS[neighbor_indices]
    mass_coeffs, _, _, _ = np.linalg.lstsq(X, mass_neighbors, rcond=None)
    
    return {
        'flux_coeffs': flux_coeffs,
        'logL_coeffs': logL_coeffs,
        'mass_coeffs': mass_coeffs,
        'teff_norm': (teff_center, teff_scale),
        'logg_norm': (logg_center, logg_scale),
    }


def evaluate_psm_flux(psm_data, teff, logg):
    """
    Evaluate the PSM to get intrinsic flux at given (Teff, logg).
    
    Returns
    -------
    flux : ndarray, shape (n_wavelengths,)
        Intrinsic flux (linear, not log)
    """
    teff_norm = psm_data['teff_norm']
    logg_norm = psm_data['logg_norm']
    coeffs = psm_data['flux_coeffs']
    
    T_norm = (teff - teff_norm[0]) / teff_norm[1]
    g_norm = (logg - logg_norm[0]) / logg_norm[1]
    
    terms = np.array([1, T_norm, g_norm, T_norm**2, g_norm**2, T_norm * g_norm])
    log_flux = coeffs @ terms
    
    return 10**log_flux


def evaluate_psm_auxiliary(psm_data, teff, logg):
    """
    Evaluate the PSM to get interpolated auxiliary quantities.
    
    Parameters
    ----------
    psm_data : dict
        PSM coefficients from build_psm_coefficients
    teff, logg : float
        Stellar parameters
    
    Returns
    -------
    dict with:
        'logL': interpolated log(L/L_sun)
        'Mass': interpolated mass (M_sun)
        'R_Rsun': radius derived from Stefan-Boltzmann (R_sun)
    """
    teff_norm = psm_data['teff_norm']
    logg_norm = psm_data['logg_norm']
    
    T_norm = (teff - teff_norm[0]) / teff_norm[1]
    g_norm = (logg - logg_norm[0]) / logg_norm[1]
    
    terms = np.array([1, T_norm, g_norm, T_norm**2, g_norm**2, T_norm * g_norm])
    
    # Interpolate logL
    logL = np.dot(psm_data['logL_coeffs'], terms)
    
    # Interpolate Mass
    mass = np.dot(psm_data['mass_coeffs'], terms)
    
    # Derive R from Stefan-Boltzmann: L = 4πR²σT⁴
    # R² = L / (4πσT⁴)
    # R = sqrt(L / (4πσT⁴))
    L_cgs = 10**logL * L_SUN  # erg/s
    R_cgs = np.sqrt(L_cgs / (4 * np.pi * SIGMA_SB * teff**4))
    R_Rsun = R_cgs / R_SUN
    
    return {
        'logL': logL,
        'Mass': mass,
        'R_Rsun': R_Rsun,
    }


print("PSM functions defined (with auxiliary quantity interpolation).")
print("  - find_neighbors_in_grid: finds 3x3 (or smaller at edges) neighbor grid")
print("  - build_psm_coefficients: fits 2D quadratic to log(flux), logL, Mass")
print("  - evaluate_psm_flux: interpolates flux for arbitrary (Teff, logg)")
print("  - evaluate_psm_auxiliary: interpolates logL, Mass; derives R_Rsun via Stefan-Boltzmann")

PSM functions defined (with auxiliary quantity interpolation).
  - find_neighbors_in_grid: finds 3x3 (or smaller at edges) neighbor grid
  - build_psm_coefficients: fits 2D quadratic to log(flux), logL, Mass
  - evaluate_psm_flux: interpolates flux for arbitrary (Teff, logg)
  - evaluate_psm_auxiliary: interpolates logL, Mass; derives R_Rsun via Stefan-Boltzmann


In [12]:
# ========================== FITTING FUNCTION (v4: PSM) ==========================

def fit_single_star_psm(sid, plx, plx_err, gmag, 
                        Teff_A23=np.nan, A_V_W25=np.nan, A_V_W25_err=np.nan):
    """
    Fit a single star using brute-force grid search + PSM refinement.
    
    Stage 1: Full brute-force search over all models (like v2)
    Stage 2: Build local PSM around best model and optimize continuously
    Stage 3: Interpolate auxiliary quantities (logL, Mass, R_Rsun) at PSM solution
    
    Returns
    -------
    dict with both discrete and refined fit results
    """
    file = SPECTRA_DIR / f"{sid}.fits"
    if not file.exists():
        return None
    
    with fits.open(file) as h:
        d = h[1].data
        w_nm = d['wavelength'].flatten()
        f    = d['flux'].flatten() * 1e2
        err  = d['flux_error'].flatten() * 1e2 if 'flux_error' in d.names else f*0.01
    
    err = np.sqrt(err**2 + (SYSTEMATIC_FLOOR*f)**2)
    w_A = w_nm * 10
    mask = (w_A >= 3400) & (w_A <= 9000)
    n_mask = mask.sum()
    if n_mask < 10:
        return None
    
    weights = wavelength_weights(w_nm[mask])
    
    # Prior flags
    use_av_prior = USE_AV_PRIOR and np.isfinite(A_V_W25) and np.isfinite(A_V_W25_err) and A_V_W25_err > 0
    use_teff_prior = USE_TEFF_PRIOR and np.isfinite(Teff_A23) and Teff_A23 < TEFF_PRIOR_TEFF_MAX
    sigma_av = A_V_W25_err if use_av_prior else 1.0
    sigma_teff = TEFF_PRIOR_SIGMA
    
    # Pre-compute Teff prior for all models
    if use_teff_prior:
        chi2_teff_all = TEFF_PRIOR_WEIGHT * ((MODEL_TEFF - Teff_A23) / sigma_teff)**2
    else:
        chi2_teff_all = np.zeros(n_models)
    
    f_obs = f[mask]
    err_obs = err[mask]
    err_obs_sq = err_obs**2
    
    # ========================================================================
    # STAGE 1: Brute-force search (same as v2)
    # ========================================================================
    
    def evaluate_av_vectorized(av_val):
        """Find best chi2 across all models and Rv values for a given Av."""
        best_chi2 = np.inf
        best_result = None
        
        for i_rv, Rv in enumerate(R_V_GRID):
            extinction_factor = 10**(-0.4 * EXT_ARRAY[i_rv, :] * av_val)
            flux_reddened_all = MODEL_FLUX_ALL * extinction_factor[np.newaxis, :]
            flux_reddened_masked = flux_reddened_all[:, mask]
            
            numerator = np.sum(f_obs[np.newaxis, :] * flux_reddened_masked / err_obs_sq[np.newaxis, :], axis=1)
            denominator = np.sum(flux_reddened_masked**2 / err_obs_sq[np.newaxis, :], axis=1)
            scale_all = numerator / denominator
            
            dist_all = np.where(scale_all > 0, 10.0 / np.sqrt(scale_all), np.inf)
            
            residuals = f_obs[np.newaxis, :] - scale_all[:, np.newaxis] * flux_reddened_masked
            chi2_spec_all = np.sum(weights[np.newaxis, :] * (residuals / err_obs[np.newaxis, :])**2, axis=1)
            
            if plx > 0 and plx_err > 0 and np.isfinite(plx_err):
                parallax_pred = 1000.0 / dist_all
                chi2_plx_all = ((parallax_pred - plx) / plx_err)**2
                chi2_plx_all = np.where(np.isfinite(chi2_plx_all), chi2_plx_all, 0)
            else:
                chi2_plx_all = np.zeros(n_models)
            
            if use_av_prior:
                chi2_av = AV_PRIOR_WEIGHT * ((av_val - A_V_W25) / sigma_av)**2
            else:
                chi2_av = 0.0
            
            chi2_tot_all = chi2_spec_all + chi2_plx_all + chi2_av + chi2_teff_all
            
            best_idx = np.argmin(chi2_tot_all)
            if chi2_tot_all[best_idx] < best_chi2:
                best_chi2 = chi2_tot_all[best_idx]
                best_result = {
                    'model_idx': best_idx,
                    'Teff': MODEL_TEFF[best_idx],
                    'logg': MODEL_LOGG[best_idx],
                    'logL': MODEL_LOGL[best_idx],
                    'Mass': MODEL_MASS[best_idx],
                    'Mass_std': MODEL_MASS_STD[best_idx],
                    'R_Rsun': MODEL_R_RSUN[best_idx],
                    'source': MODEL_SOURCE[best_idx],
                    'filename': MODEL_FILENAME[best_idx],
                    'A_V': av_val,
                    'R_V': Rv,
                    'scale': scale_all[best_idx],
                    'dist_pc': dist_all[best_idx],
                    'chi2_tot': chi2_tot_all[best_idx],
                    'chi2_spec': chi2_spec_all[best_idx],
                    'chi2_plx': chi2_plx_all[best_idx],
                    'chi2_av_prior': chi2_av,
                    'chi2_teff_prior': chi2_teff_all[best_idx],
                }
        
        return best_chi2, best_result
    
    # Coarse grid search in A_V
    av_coarse = np.arange(0, 6.05, 0.5)
    chi2_coarse = []
    results_coarse = []
    
    for av_val in av_coarse:
        chi2, result = evaluate_av_vectorized(av_val)
        chi2_coarse.append(chi2)
        results_coarse.append(result)
    
    chi2_coarse = np.array(chi2_coarse)
    best_coarse_idx = np.argmin(chi2_coarse)
    
    # Refine A_V with iterations
    av_center = av_coarse[best_coarse_idx]
    search_radius = 0.5
    
    for iteration in range(3):
        av_refined = np.linspace(max(0, av_center - search_radius), 
                                 min(6.0, av_center + search_radius), 5)
        chi2_refined = []
        results_refined = []
        
        for av_val in av_refined:
            chi2, result = evaluate_av_vectorized(av_val)
            chi2_refined.append(chi2)
            results_refined.append(result)
        
        chi2_refined = np.array(chi2_refined)
        best_refined_idx = np.argmin(chi2_refined)
        
        if 0 < best_refined_idx < len(av_refined) - 1:
            idx_range = [best_refined_idx - 1, best_refined_idx, best_refined_idx + 1]
            av_pts = av_refined[idx_range]
            chi_pts = chi2_refined[idx_range]
            try:
                coeffs = np.polyfit(av_pts, chi_pts, 2)
                a, b, c = coeffs
                if a > 0:
                    av_parabolic = -b / (2 * a)
                    if av_pts[0] <= av_parabolic <= av_pts[2]:
                        av_center = av_parabolic
                    else:
                        av_center = av_refined[best_refined_idx]
                else:
                    av_center = av_refined[best_refined_idx]
            except:
                av_center = av_refined[best_refined_idx]
        else:
            av_center = av_refined[best_refined_idx]
        
        av_center = np.clip(av_center, 0, 6.0)
        search_radius = search_radius / 2.5
    
    # Final discrete evaluation
    chi2_final, discrete_result = evaluate_av_vectorized(av_center)
    
    if chi2_coarse[best_coarse_idx] < chi2_final:
        discrete_result = results_coarse[best_coarse_idx]
    
    # ========================================================================
    # STAGE 2: PSM Refinement
    # ========================================================================
    
    best_model_idx = discrete_result['model_idx']
    
    # Find neighbors
    neighbor_indices, teff_range, logg_range = find_neighbors_in_grid(best_model_idx, n_neighbors=1)
    
    # Check if we have enough neighbors for meaningful PSM
    if len(neighbor_indices) < 4:
        # Not enough neighbors, return discrete result only
        discrete_result['psm_refined'] = False
        discrete_result['Teff_psm'] = discrete_result['Teff']
        discrete_result['logg_psm'] = discrete_result['logg']
        discrete_result['A_V_psm'] = discrete_result['A_V']
        discrete_result['R_V_psm'] = discrete_result['R_V']
        discrete_result['chi2_psm'] = discrete_result['chi2_tot']
        discrete_result['logL_psm'] = discrete_result['logL']
        discrete_result['Mass_psm'] = discrete_result['Mass']
        discrete_result['R_Rsun_psm'] = discrete_result['R_Rsun']
        discrete_result['flux_mod'] = apply_reddening(
            gaia_wavelength_A, 
            MODEL_FLUX_ALL[best_model_idx, :] * discrete_result['scale'],
            discrete_result['A_V'], discrete_result['R_V']
        )
        discrete_result['w_obs'] = w_nm
        discrete_result['f_obs'] = f
        discrete_result['err_obs'] = err
        discrete_result['used_av_prior'] = use_av_prior
        discrete_result['used_teff_prior'] = use_teff_prior
        discrete_result['n_psm_neighbors'] = len(neighbor_indices)
        return discrete_result
    
    # Build PSM coefficients (now includes logL and Mass)
    psm_data = build_psm_coefficients(neighbor_indices, teff_range, logg_range)
    
    # Define chi2 function for continuous optimization
    def chi2_continuous(params):
        """Compute chi2 for continuous (Teff, logg, A_V, R_V)."""
        teff, logg, av, rv = params
        
        # Clamp R_V to valid range
        rv = np.clip(rv, 2.3, 5.6)
        av = max(0, av)
        
        # Get intrinsic flux from PSM
        try:
            flux_intrinsic = evaluate_psm_flux(psm_data, teff, logg)
        except:
            return 1e10
        
        # Apply extinction
        ext_curve = get_extinction_curve(rv)
        flux_reddened = flux_intrinsic * 10**(-0.4 * av * ext_curve)
        flux_reddened_masked = flux_reddened[mask]
        
        # Compute optimal scale
        num = np.sum(f_obs * flux_reddened_masked / err_obs_sq)
        den = np.sum(flux_reddened_masked**2 / err_obs_sq)
        if den <= 0:
            return 1e10
        scale = num / den
        if scale <= 0:
            return 1e10
        
        dist_pc = 10.0 / np.sqrt(scale)
        
        # Spectral chi2
        residuals = f_obs - scale * flux_reddened_masked
        chi2_spec = np.sum(weights * (residuals / err_obs)**2)
        
        # Parallax chi2
        if plx > 0 and plx_err > 0 and np.isfinite(plx_err):
            parallax_pred = 1000.0 / dist_pc
            chi2_plx = ((parallax_pred - plx) / plx_err)**2
        else:
            chi2_plx = 0.0
        
        # A_V prior
        if use_av_prior:
            chi2_av = AV_PRIOR_WEIGHT * ((av - A_V_W25) / sigma_av)**2
        else:
            chi2_av = 0.0
        
        # Teff prior
        if use_teff_prior:
            chi2_teff = TEFF_PRIOR_WEIGHT * ((teff - Teff_A23) / sigma_teff)**2
        else:
            chi2_teff = 0.0
        
        return chi2_spec + chi2_plx + chi2_av + chi2_teff
    
    # Initial guess from discrete result
    x0 = [discrete_result['Teff'], discrete_result['logg'], 
          discrete_result['A_V'], discrete_result['R_V']]
    
    # Bounds: stay within local PSM region for Teff/logg, physical for A_V/R_V
    bounds = [
        (teff_range[0], teff_range[1]),
        (logg_range[0], logg_range[1]),
        (0.0, 6.0),
        (2.3, 5.6)
    ]
    
    # Optimize
    try:
        opt_result = minimize(chi2_continuous, x0, method='L-BFGS-B', bounds=bounds,
                             options={'maxiter': 100, 'ftol': 1e-6})
        
        if opt_result.success or opt_result.fun < discrete_result['chi2_tot']:
            teff_psm, logg_psm, av_psm, rv_psm = opt_result.x
            chi2_psm = opt_result.fun
            psm_refined = True
        else:
            teff_psm = discrete_result['Teff']
            logg_psm = discrete_result['logg']
            av_psm = discrete_result['A_V']
            rv_psm = discrete_result['R_V']
            chi2_psm = discrete_result['chi2_tot']
            psm_refined = False
    except:
        teff_psm = discrete_result['Teff']
        logg_psm = discrete_result['logg']
        av_psm = discrete_result['A_V']
        rv_psm = discrete_result['R_V']
        chi2_psm = discrete_result['chi2_tot']
        psm_refined = False
    
    # ========================================================================
    # STAGE 3: Interpolate auxiliary quantities at PSM solution
    # ========================================================================
    
    if psm_refined:
        aux = evaluate_psm_auxiliary(psm_data, teff_psm, logg_psm)
        logL_psm = aux['logL']
        mass_psm = aux['Mass']
        r_rsun_psm = aux['R_Rsun']
    else:
        logL_psm = discrete_result['logL']
        mass_psm = discrete_result['Mass']
        r_rsun_psm = discrete_result['R_Rsun']
    
    # Compute final model spectrum for plotting (using PSM parameters)
    flux_intrinsic_psm = evaluate_psm_flux(psm_data, teff_psm, logg_psm)
    ext_curve_psm = get_extinction_curve(rv_psm)
    flux_reddened_psm = flux_intrinsic_psm * 10**(-0.4 * av_psm * ext_curve_psm)
    
    # Compute scale for PSM result
    flux_reddened_psm_masked = flux_reddened_psm[mask]
    num = np.sum(f_obs * flux_reddened_psm_masked / err_obs_sq)
    den = np.sum(flux_reddened_psm_masked**2 / err_obs_sq)
    scale_psm = num / den if den > 0 else discrete_result['scale']
    dist_psm = 10.0 / np.sqrt(scale_psm) if scale_psm > 0 else discrete_result['dist_pc']
    
    # Build final result
    result = discrete_result.copy()
    result['psm_refined'] = psm_refined
    result['Teff_psm'] = teff_psm
    result['logg_psm'] = logg_psm
    result['A_V_psm'] = av_psm
    result['R_V_psm'] = rv_psm
    result['chi2_psm'] = chi2_psm
    result['dist_pc_psm'] = dist_psm
    result['scale_psm'] = scale_psm
    result['logL_psm'] = logL_psm
    result['Mass_psm'] = mass_psm
    result['R_Rsun_psm'] = r_rsun_psm
    result['n_psm_neighbors'] = len(neighbor_indices)
    result['flux_mod'] = flux_reddened_psm * scale_psm
    result['w_obs'] = w_nm
    result['f_obs'] = f
    result['err_obs'] = err
    result['used_av_prior'] = use_av_prior
    result['used_teff_prior'] = use_teff_prior
    
    return result

In [13]:
# ========================== PLOTTING FUNCTION ==========================
def plot_fit(sid, fit_params, Teff_A23=np.nan, A_V_W25=np.nan):
    """Generate diagnostic plot for one star."""
    if fit_params is None:
        return

    w_nm  = fit_params['w_obs']
    f_obs = fit_params['f_obs']
    f_mod = fit_params['flux_mod']
    err   = fit_params['err_obs']

    w_A   = w_nm * 10.0
    mask  = (w_A >= 3400) & (w_A <= 9000)

    # Use PSM parameters if available
    if fit_params.get('psm_refined', False):
        dist_pc = fit_params['dist_pc_psm']
        teff_fit = fit_params['Teff_psm']
        logg_fit = fit_params['logg_psm']
        av_fit = fit_params['A_V_psm']
        rv_fit = fit_params['R_V_psm']
        chi2_val = fit_params['chi2_psm']
        method_str = "PSM"
    else:
        dist_pc = fit_params['dist_pc']
        teff_fit = fit_params['Teff']
        logg_fit = fit_params['logg']
        av_fit = fit_params['A_V']
        rv_fit = fit_params['R_V']
        chi2_val = fit_params['chi2_tot']
        method_str = "grid"

    dist_str = f"{dist_pc:.0f}" if np.isfinite(dist_pc) and dist_pc > 0 else "???"

    n_spec = mask.sum()
    n_plx  = 1 if fit_params.get('chi2_plx', 0) > 0 else 0
    dof    = n_spec + n_plx - 1
    chi2_red = chi2_val / dof if dof > 0 else 999.99

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                   gridspec_kw={'height_ratios': [3, 1]})

    ax1.plot(w_nm, f_obs, 'ko', ms=3, alpha=0.7, label='Gaia BP/RP')
    
    label = f"Teff={teff_fit/1000:.1f} kK  log g={logg_fit:.2f} ({method_str})\n"
    label += f"A_V={av_fit:.2f}  R_V={rv_fit:.2f}"
    
    prior_parts = []
    if np.isfinite(A_V_W25):
        prior_parts.append(f"$A_V$(W25)={A_V_W25:.2f}")
    if np.isfinite(Teff_A23):
        prior_parts.append(f"$T_{{eff}}$(A23)={Teff_A23:.0f}K")
    if prior_parts:
        label += "\n" + "  ".join(prior_parts)
    
    ax1.plot(w_nm, f_mod, 'r-', lw=2, label=label)

    # Rayleigh-Jeans reference
    w_ref = 600.0
    f_ref = np.interp(w_ref, w_nm, f_mod, left=np.nan, right=np.nan)
    if np.isfinite(f_ref):
        f_ref *= 0.9
        w_rj = np.linspace(340, 1050, 200)
        ax1.plot(w_rj, f_ref * (w_ref/w_rj)**4, '--', color='blue', lw=1.5,
                 alpha=0.8, label='Rayleigh-Jeans')

    ax1.set_yscale('log')
    ax1.set_xlim(330, 1050)
    ax1.set_ylabel('Flux (erg s$^{-1}$ cm$^{-2}$ A$^{-1}$)')
    ax1.legend(fontsize=10)
    ax1.set_title(f"source_id = {sid} | d = {dist_str} pc | chi2_red = {chi2_red:.2f}")

    resid = (f_obs - f_mod) / err
    ax2.plot(w_nm, resid, 'ko', ms=3, alpha=0.7)
    ax2.axhline(0, color='red', ls='--', lw=1)
    ax2.axhspan(-3, 3, color='gray', alpha=0.1)
    ax2.set_xlabel('Wavelength (nm)')
    ax2.set_ylabel('Residual (sigma)')
    ax2.set_xlim(330, 1050)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    outfile = PLOT_DIR / f"fit_{sid}.png"
    plt.savefig(outfile, dpi=150, bbox_inches='tight')
    plt.close()

In [14]:
# ========================== MAIN LOOP ==========================

n_to_fit = len(source_ids) if N_MAX_FIT is None else min(N_MAX_FIT, len(source_ids))
print(f"Fitting {n_to_fit} stars" + (" (TEST MODE)" if N_MAX_FIT else "") + "\n")
print(f"Using PSM (Polynomial Spectral Model) refinement (v4)\n")
print(f"Auxiliary quantities (logL, Mass, R_Rsun) are PSM-interpolated\n")

results = []
n_with_av_prior = 0
n_with_teff_prior = 0
n_psm_refined = 0

t_start = time.time()

for i, sid in enumerate(source_ids[:n_to_fit], 1):
    print(f"[{i:4d}/{n_to_fit}] {sid} ", end='')
    
    teff_a23 = Teff_A23_catalog[i-1]
    av_w25 = A_V_W25_catalog[i-1]
    av_w25_err = A_V_W25_err_catalog[i-1]
    
    t0 = time.time()
    res = fit_single_star_psm(sid, parallax[i-1], parallax_error[i-1], gmag[i-1],
                              Teff_A23=teff_a23, A_V_W25=av_w25, A_V_W25_err=av_w25_err)
    t1 = time.time()
    
    if not res:
        print("no spectrum")
        continue
    
    # Use PSM values if refined, else discrete
    if res.get('psm_refined', False):
        teff_out = res['Teff_psm']
        logg_out = res['logg_psm']
        av_out = res['A_V_psm']
        rv_out = res['R_V_psm']
        dist_out = res['dist_pc_psm']
        chi2_out = res['chi2_psm']
        logL_out = res['logL_psm']
        mass_out = res['Mass_psm']
        r_rsun_out = res['R_Rsun_psm']
        n_psm_refined += 1
    else:
        teff_out = res['Teff']
        logg_out = res['logg']
        av_out = res['A_V']
        rv_out = res['R_V']
        dist_out = res['dist_pc']
        chi2_out = res['chi2_tot']
        logL_out = res['logL']
        mass_out = res['Mass']
        r_rsun_out = res['R_Rsun']
    
    A_G = calculate_A_G(av_out, rv_out)
    M_G = gmag[i-1] - 5*np.log10(dist_out) + 5 - A_G if np.isfinite(dist_out) else np.nan
    
    if res.get('used_av_prior', False):
        n_with_av_prior += 1
    if res.get('used_teff_prior', False):
        n_with_teff_prior += 1
    
    result_dict = {
        'source_id': sid,
        # Primary fit results (PSM-interpolated if refined, else discrete)
        'Teff_fit': teff_out,
        'logg_fit': logg_out,
        'A_V_fit': av_out,
        'R_V_fit': rv_out,
        'distance_pc_fit': dist_out,
        'M_G_fit': M_G,
        'chi2_red_fit': chi2_out / (len(res['w_obs']) - 1),
        'logL_fit': logL_out,
        'Mass_fit': mass_out,
        'R_Rsun_fit': r_rsun_out,
        # Discrete grid results (for comparison)
        'Teff_grid': res['Teff'],
        'logg_grid': res['logg'],
        'A_V_grid': res['A_V'],
        'R_V_grid': res['R_V'],
        'distance_pc_grid': res['dist_pc'],
        'chi2_grid': res['chi2_tot'],
        'logL_grid': res['logL'],
        'Mass_grid': res['Mass'],
        'R_Rsun_grid': res['R_Rsun'],
        # Model info
        'model_source': res.get('source', 'unknown'),
        'model_filename': res.get('filename', ''),
        'Mass_std_grid': res.get('Mass_std', np.nan),
        # PSM info
        'psm_refined': res.get('psm_refined', False),
        'n_psm_neighbors': res.get('n_psm_neighbors', 0),
        # Prior info
        'used_av_prior': res.get('used_av_prior', False),
        'used_teff_prior': res.get('used_teff_prior', False),
        'chi2_av_prior': res.get('chi2_av_prior', 0),
        'chi2_teff_prior': res.get('chi2_teff_prior', 0),
    }
    results.append(result_dict)
    
    plot_fit(sid, res, Teff_A23=teff_a23, A_V_W25=av_w25)
    
    # Print summary
    psm_str = " [PSM]" if res.get('psm_refined', False) else ""
    prior_str = ""
    if res.get('used_av_prior', False):
        prior_str += f" [AV:{av_w25:.2f}]"
    mass_str = f" M={mass_out:.1f}Msun" if np.isfinite(mass_out) else ""
    print(f"Teff={teff_out/1000:.1f}kK logg={logg_out:.2f} A_V={av_out:.2f} d={dist_out:.0f}pc{mass_str}{psm_str}{prior_str} ({t1-t0:.2f}s)")

t_end = time.time()

# Create fit results DataFrame
fit_df = pd.DataFrame(results)

# Merge with original catalog to preserve all input columns
df = catalog_df.merge(fit_df, on='source_id', how='inner')

# Save output
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nFinished! Saved to {OUTPUT_CSV}")
print(f"Fitted {len(results)} stars successfully")
print(f"\nTotal time: {t_end - t_start:.1f}s ({(t_end - t_start)/len(results):.2f}s per star)")
print(f"\nPSM refinement: {n_psm_refined}/{len(results)} stars ({100*n_psm_refined/len(results):.1f}%)")
print(f"\nPrior usage:")
print(f"  A_V prior applied: {n_with_av_prior} stars")
print(f"  Teff prior applied: {n_with_teff_prior} stars")

Fitting 280 stars

Using PSM (Polynomial Spectral Model) refinement (v4)

Auxiliary quantities (logL, Mass, R_Rsun) are PSM-interpolated

[   1/280] 2178808979101816192 Teff=15.0kK logg=3.00 A_V=1.79 d=1237pc M=8.1Msun [PSM] [AV:1.51] (0.80s)
[   2/280] 4069469560083491712 no spectrum
[   3/280] 3127680019142800128 no spectrum
[   4/280] 6044420729667868928 no spectrum
[   5/280] 4093346863913375232 Teff=35.0kK logg=3.37 A_V=2.00 d=2003pc M=60.5Msun [PSM] [AV:1.99] (0.55s)
[   6/280] 1963714268134133120 Teff=19.0kK logg=2.74 A_V=0.41 d=2150pc M=18.7Msun [PSM] [AV:0.39] (0.52s)
[   7/280] 463932269854231168 Teff=35.0kK logg=3.59 A_V=2.37 d=2577pc M=35.9Msun [PSM] [AV:2.05] (0.70s)
[   8/280] 532804028948316672 Teff=24.0kK logg=3.40 A_V=1.41 d=1130pc M=13.9Msun [PSM] [AV:1.43] (0.67s)
[   9/280] 3337904172766584064 Teff=19.0kK logg=3.37 A_V=0.26 d=417pc M=9.7Msun [PSM] [AV:0.22] (0.61s)
[  10/280] 537106933343756800 Teff=22.0kK logg=3.57 A_V=0.78 d=1012pc M=11.0Msun [PSM] [AV:0.87] (0.67

In [15]:
# Summary statistics
print("="*60)
print("FIT SUMMARY (v4 - PSM)")
print("="*60)
print(f"\nTotal stars fitted: {len(df)}")
print(f"PSM refined: {df['psm_refined'].sum()} ({100*df['psm_refined'].mean():.1f}%)")

print(f"\nTeff distribution (PSM/grid):")
print(f"  Min: {df['Teff_fit'].min():.0f} K")
print(f"  Max: {df['Teff_fit'].max():.0f} K")
print(f"  Median: {df['Teff_fit'].median():.0f} K")

# Compare PSM vs grid values for refined stars
refined = df[df['psm_refined']]
if len(refined) > 0:
    print(f"\nPSM refinement statistics (N={len(refined)}):")
    print(f"  Teff shift: median={np.median(refined['Teff_fit'] - refined['Teff_grid']):.0f} K, "
          f"std={np.std(refined['Teff_fit'] - refined['Teff_grid']):.0f} K")
    print(f"  logg shift: median={np.median(refined['logg_fit'] - refined['logg_grid']):.3f}, "
          f"std={np.std(refined['logg_fit'] - refined['logg_grid']):.3f}")
    print(f"  A_V shift: median={np.median(refined['A_V_fit'] - refined['A_V_grid']):.3f} mag, "
          f"std={np.std(refined['A_V_fit'] - refined['A_V_grid']):.3f} mag")
    print(f"  Chi2 improvement: median={np.median(refined['chi2_grid'] - refined['chi2_red_fit'] * (len(df['Teff_fit'])-1)):.1f}")

print(f"\nA_V distribution:")
print(f"  Min: {df['A_V_fit'].min():.2f} mag")
print(f"  Max: {df['A_V_fit'].max():.2f} mag")
print(f"  Median: {df['A_V_fit'].median():.2f} mag")

print(f"\nR_V distribution:")
print(df['R_V_fit'].describe())

print(f"\nModel sources:")
print(df['model_source'].value_counts())

print(f"\nPriors applied:")
print(f"  A_V prior: {df['used_av_prior'].sum()} ({100*df['used_av_prior'].mean():.1f}%)")
print(f"  Teff prior: {df['used_teff_prior'].sum()} ({100*df['used_teff_prior'].mean():.1f}%)")

FIT SUMMARY (v4 - PSM)

Total stars fitted: 230
PSM refined: 230 (100.0%)

Teff distribution (PSM/grid):
  Min: 9000 K
  Max: 43000 K
  Median: 22000 K

PSM refinement statistics (N=230):
  Teff shift: median=0 K, std=6 K
  logg shift: median=0.001, std=0.072
  A_V shift: median=-0.000 mag, std=0.064 mag
  Chi2 improvement: median=67.5

A_V distribution:
  Min: 0.00 mag
  Max: 5.98 mag
  Median: 1.24 mag

R_V distribution:
count    230.000000
mean       3.337594
std        1.031790
min        2.300000
25%        2.591308
50%        3.018429
75%        3.688103
max        5.600000
Name: R_V_fit, dtype: float64

Model sources:
model_source
PoWR       173
ATLAS       52
PHOENIX      5
Name: count, dtype: int64

Priors applied:
  A_V prior: 226 (98.3%)
  Teff prior: 0 (0.0%)
